# 🧠 Delentia AI OS 1+4 Pillars — Google Colab Fine-tuning

This notebook provides a complete guide for fine-tuning the 4 specialized LoRA adapters in the **Delentia OS 1+4 Pillar Architecture** (Executor, Router, Guardian, Scribe) using Unsloth (QLoRA).

---

## Pillars Training Roadmap

| Phase | Adapter | Task Type | Script | Training Command |
| :--- | :--- | :--- | :--- | :--- |
| **1** | **The Executor** (`slm-jitna-agentic`) | Causal LM (JSON API) | `finetune.py` | `python training/finetune.py --pillar executor` |
| **2** | **The Router** (`slm-jitna-router`) | Seq Classification | `finetune_classifier.py` | `python training/finetune_classifier.py` |
| **3** | **The Guardian** (`slm-jitna-guardian`) | Causal LM (Safety Shield) | `finetune.py` | `python training/finetune.py --pillar guardian` |
| **4** | **The Scribe** (`slm-jitna-scribe`) | Causal LM (Compression) | `finetune.py` | `python training/finetune.py --pillar scribe` |

---

## Shared Drive Checkpoint Strategy

1. **Google Drive Mount**: Checkpoints are automatically synced to Google Drive.
2. **Quota Optimization**: If your GPU quota runs out on your primary Google account, share the Google Drive folder with a secondary account, open the notebook, remount, and resume training the next adapter.

In [ ]:
# ─── Cell 1: Mount Google Drive + clone repos ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted successfully.')
except Exception as e:
    print(f'⚠️ Google Drive mount skipped or failed: {e}')
    print('Proceeding with local runtime storage.')

import os, subprocess, sys

# Map Colab secrets to environment variables
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY") or userdata.get("KAGGLE_k") or ""
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME") or "delentialabs"
    print("✅ Environment credentials mapped successfully.")
except Exception as e:
    print(f"Credential mapping warning: {e}")

REPO_URL = 'https://github.com/delentia-labs/Delentia-AI-SLM.git'
REPO_DIR = '/content/Delentia-AI-SLM'
OS_URL = 'https://github.com/delentia-labs/Delentia-OS.git'
OS_DIR = '/content/Delentia-OS'

# Clone SLM Training repo
if not os.path.exists(REPO_DIR):
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print('SLM repo already exists — pulled latest:', result.stdout.strip())

# Clone OS repo
if not os.path.exists(OS_DIR):
    result = subprocess.run(['git', 'clone', OS_URL, OS_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(['git', '-C', OS_DIR, 'pull'], capture_output=True, text=True)
    print('OS repo already exists — pulled latest:', result.stdout.strip())

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ─── Cell 2: Install dependencies ────────────────────────────────────────────
import subprocess, sys, torch

if not torch.cuda.is_available():
    print('⚠️ WARNING: GPU runtime is not active! please change Colab runtime to GPU (Runtime -> Change runtime type)')
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU detected: {gpu_name}')

# Install Unsloth + training deps
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--quiet'
])

# Install local Delentia OS SDK package
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-e', '/content/Delentia-OS', '--quiet'
])

# Install project requirements
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '--quiet'
])

print('✅ All dependencies installed.')

In [ ]:
# ─── Cell 3: Compile datasets for all 4 pillars ──────────────────────────────
import subprocess, sys
print("Synthesizing datasets...")
subprocess.run([sys.executable, 'datasets/scripts/generate_executor_dataset.py'], check=True)
subprocess.run([sys.executable, 'datasets/scripts/generate_router_dataset.py'], check=True)
subprocess.run([sys.executable, 'datasets/scripts/generate_guardian_dataset.py'], check=True)
subprocess.run([sys.executable, 'datasets/scripts/generate_scribe_dataset.py'], check=True)
print("✅ Datasets generated.")

In [ ]:
# ─── Cell 4: Train Adapter #1 — The Executor (JSON/Tool Calling) ─────────────
import subprocess, sys
print("Training Executor...")
process = subprocess.Popen([sys.executable, 'training/finetune.py', '--pillar', 'executor'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Executor training failed with code {process.returncode}")
print("✅ Executor adapter training complete.")

In [ ]:
# ─── Cell 5: Train Adapter #2 — The Router (Sequence Classifier) ─────────────
import subprocess, sys
print("Training Router Classifier...")
process = subprocess.Popen([sys.executable, 'training/finetune_classifier.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Router training failed with code {process.returncode}")
print("✅ Router adapter training complete.")

In [ ]:
# ─── Cell 6: Train Adapter #3 — The Guardian (Safety Shield) ─────────────────
import subprocess, sys
print("Training Guardian...")
process = subprocess.Popen([sys.executable, 'training/finetune.py', '--pillar', 'guardian'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Guardian training failed with code {process.returncode}")
print("✅ Guardian adapter training complete.")

In [ ]:
# ─── Cell 7: Train Adapter #4 — The Scribe (Context Compression) ──────────────
import subprocess, sys
print("Training Scribe...")
process = subprocess.Popen([sys.executable, 'training/finetune.py', '--pillar', 'scribe'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Scribe training failed with code {process.returncode}")
print("✅ Scribe adapter training complete.")

In [ ]:
# ─── Cell 8: Model Evaluations ──────────────────────────────────────────────
import subprocess, sys
print("Running evaluations for all 4 pillars...")
subprocess.run([sys.executable, 'training/evaluate.py', '--pillar', 'executor'], check=True)
subprocess.run([sys.executable, 'training/evaluate.py', '--pillar', 'router'], check=True)
subprocess.run([sys.executable, 'training/evaluate.py', '--pillar', 'guardian'], check=True)
subprocess.run([sys.executable, 'training/evaluate.py', '--pillar', 'scribe'], check=True)
print("✅ All evaluations passed successfully.")

In [ ]:
# ─── Cell 9: GGUF Quantization Export ────────────────────────────────────────
import subprocess, sys, os
print("Exporting generative adapters to GGUF format...")
subprocess.run([sys.executable, 'training/export_gguf.py', '--pillar', 'executor'], check=True)
subprocess.run([sys.executable, 'training/export_gguf.py', '--pillar', 'guardian'], check=True)
subprocess.run([sys.executable, 'training/export_gguf.py', '--pillar', 'scribe'], check=True)

# Sync model checkpoints to Google Drive
print("Syncing model checkpoints to Google Drive...")
os.makedirs('/content/drive/MyDrive/delentia_adapters/', exist_ok=True)
subprocess.run(['cp', '-r', 'models/adapters/', '/content/drive/MyDrive/delentia_adapters/'])
print("🎉 Export and Drive backup complete!")

In [ ]:
# ─── Cell 10: Publish to HuggingFace Hub ──────────────────────────────────────
import os, glob
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    print('⚠️ HF_TOKEN not set. Add it in Colab Secrets (key icon) or set HF_TOKEN environment variable.')
else:
    login(token=hf_token)
    api = HfApi()
    
    # We upload GGUF files and configs for each pillar to their respective Hugging Face paths
    for pillar in ["executor", "guardian", "scribe"]:
        repo_id = f"Delentia/delentia-slm-jitna-{pillar}"
        api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True, private=False)
        
        # Upload GGUF files matching the pillar name
        gguf_files = glob.glob(f"models/gguf/*{pillar}*.gguf")
        for gguf_file in gguf_files:
            filename = os.path.basename(gguf_file)
            print(f"Uploading {filename} to {repo_id}...")
            api.upload_file(
                path_or_fileobj=gguf_file,
                path_in_repo=f"gguf/{filename}",
                repo_id=repo_id,
                repo_type='model',
            )
            print(f"  ✅ Uploaded {filename}")
            
    # Router classification adapter
    router_repo = "Delentia/delentia-slm-jitna-router"
    api.create_repo(repo_id=router_repo, repo_type='model', exist_ok=True, private=False)
    # Upload adapter checkpoint files directly for sequence classification
    adapter_files = glob.glob("models/adapters/jitna_router_v1/*")
    for file_path in adapter_files:
        if os.path.isfile(file_path):
            fname = os.path.basename(file_path)
            print(f"Uploading {fname} to {router_repo}...")
            api.upload_file(
                path_or_fileobj=file_path,
                path_in_repo=fname,
                repo_id=router_repo,
                repo_type='model',
            )
    print("🎉 Publishing complete!")